In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# 1. STANDARD RANDOM FOREST PIPELINE
# ------------------------------------------------------------

def run_standard_random_forest(df, 
                              target_column='label',
                              test_size=0.2,
                              random_state=42,
                              external_test_df=None,
                              preserve_order=True):
    """
    Standard Random Forest pipeline
    Supports both internal train-test split and external test set
    """
    
    print("="*80)
    print("STANDARD RANDOM FOREST PIPELINE")
    print("="*80)
    
    # --------------------------------------------------------
    # 1. Data Preparation
    # --------------------------------------------------------
    df = df.drop(columns=['IdleTime', 'DstWin'])

    df_processed = df.copy()
    
    if target_column not in df_processed.columns:
        available_cols = df_processed.columns.tolist()
        print(f"Warning: Target column '{target_column}' not found.")
        print(f"Available columns: {available_cols}")
        
        if 'label' in available_cols:
            target_column = 'label'
            print(f"Using 'label' as target column instead.")
        else:
            raise ValueError(f"Target column not found. Available: {available_cols}")
    
    all_features = [col for col in df_processed.columns if col != target_column]
    
    if not all_features:
        raise ValueError("No feature columns found!")
    
    print(f"\n[1] DATA PREPARATION")
    print(f"    Target column: {target_column}")
    print(f"    Number of features: {len(all_features)}")
    print(f"    Total samples: {len(df_processed)}")
    
    label_info = df_processed[target_column].value_counts().sort_index()
    n_classes = len(label_info)
    print(f"    Number of classes: {n_classes}")
    print(f"    Class distribution:")
    for label, count in label_info.items():
        percentage = (count / len(df_processed)) * 100
        print(f"      Class {label}: {count} samples ({percentage:.1f}%)")
    
    # --------------------------------------------------------
    # 2. Train-Test Split (or use external test)
    # --------------------------------------------------------
    
    print(f"\n[2] DATA SPLITTING")
    
    if external_test_df is not None:
        # Use external test set
        print(f"    Using external test set provided")
        train_df = df_processed.copy()
        test_df = external_test_df.copy()
        
        # Ensure test set has same columns as train set
        missing_cols = set(train_df.columns) - set(test_df.columns)
        if missing_cols:
            print(f"    Warning: Test set missing columns: {missing_cols}")
            for col in missing_cols:
                if col != target_column:
                    test_df[col] = 0
        
        # Keep only common columns
        common_cols = list(set(train_df.columns) & set(test_df.columns))
        train_df = train_df[common_cols]
        test_df = test_df[common_cols]
        
        print(f"    Training samples: {len(train_df)}")
        print(f"    Test samples: {len(test_df)}")
        
    else:
        # Internal train-test split
        if preserve_order:
            # Class-wise sequential split (preserves order AND label distribution)
            print(f"    Using CLASS-WISE sequential split (order preserved)")
            
            train_dfs = []
            test_dfs = []
            
            # Get unique labels
            unique_labels = df_processed[target_column].unique()
            print(f"    Splitting each of {len(unique_labels)} classes separately...")
            
            for label in unique_labels:
                # Get data for this class only
                class_data = df_processed[df_processed[target_column] == label].copy()
                n_class_samples = len(class_data)
                
                if n_class_samples > 0:
                    # Calculate split index for this class
                    split_idx = int(n_class_samples * (1 - test_size))
                    
                    # Split this class's data
                    class_train = class_data.iloc[:split_idx]
                    class_test = class_data.iloc[split_idx:]
                    
                    train_dfs.append(class_train)
                    test_dfs.append(class_test)
                    
                    print(f"      Class {label}: {n_class_samples} total -> "
                          f"{len(class_train)} train, {len(class_test)} test")
            
            # Combine all classes
            train_df = pd.concat(train_dfs, ignore_index=False).sort_index()
            test_df = pd.concat(test_dfs, ignore_index=False).sort_index()
            
            print(f"\n    Training samples: {len(train_df)}")
            print(f"    Test samples: {len(test_df)}")
            
        else:
            # Original random split with stratification
            print(f"    Using random split with stratification")
            if n_classes > 1:
                train_df, test_df = train_test_split(
                    df_processed, 
                    test_size=test_size, 
                    stratify=df_processed[target_column], 
                    random_state=random_state
                )
            else:
                print("    Warning: Only one class found. Stratification disabled.")
                train_df, test_df = train_test_split(
                    df_processed, 
                    test_size=test_size, 
                    random_state=random_state
                )
            print(f"    Training samples: {len(train_df)}")
            print(f"    Test samples: {len(test_df)}")
    
    # --------------------------------------------------------
    # 3. Data Scaling
    # --------------------------------------------------------
    
    print(f"\n[3] DATA SCALING")
    
    # Separate features and target
    X_train = train_df[all_features]
    y_train = train_df[target_column]
    X_test = test_df[all_features]
    y_test = test_df[target_column]
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    print(f"    Features scaled using StandardScaler")
    
    # --------------------------------------------------------
    # 4. Train Random Forest Model
    # --------------------------------------------------------
    
    print(f"\n[4] TRAINING RANDOM FOREST MODEL")
    
    # Initialize Random Forest classifier
    rf_classifier = RandomForestClassifier(
        n_estimators=100,
        random_state=random_state,
        class_weight='balanced' if n_classes > 1 else None,
        n_jobs=-1  # Use all available cores
    )
    
    # Train the model
    rf_classifier.fit(X_train_scaled, y_train)
    print(f"    Model trained with {rf_classifier.n_estimators} trees")
    
    # --------------------------------------------------------
    # 5. Model Evaluation
    # --------------------------------------------------------
    
    print(f"\n[5] MODEL EVALUATION")
    
    # Predictions
    y_train_pred = rf_classifier.predict(X_train_scaled)
    y_test_pred = rf_classifier.predict(X_test_scaled)
    
    # Training results
    train_results = {
        'accuracy': accuracy_score(y_train, y_train_pred),
        'f1_weighted': f1_score(y_train, y_train_pred, average='weighted', zero_division=0),
        'recall_weighted': recall_score(y_train, y_train_pred, average='weighted', zero_division=0),
        'precision_weighted': precision_score(y_train, y_train_pred, average='weighted', zero_division=0),
        'n_samples': len(y_train)
    }
    
    # Test results
    test_results = {
        'accuracy': accuracy_score(y_test, y_test_pred),
        'f1_weighted': f1_score(y_test, y_test_pred, average='weighted', zero_division=0),
        'recall_weighted': recall_score(y_test, y_test_pred, average='weighted', zero_division=0),
        'precision_weighted': precision_score(y_test, y_test_pred, average='weighted', zero_division=0),
        'confusion_matrix': confusion_matrix(y_test, y_test_pred),
        'classification_report': classification_report(y_test, y_test_pred, digits=3),
        'n_samples': len(y_test)
    }
    
    # Feature importances
    feature_importances = pd.DataFrame({
        'feature': all_features,
        'importance': rf_classifier.feature_importances_
    }).sort_values('importance', ascending=False)
    
    # --------------------------------------------------------
    # 6. Display Results
    # --------------------------------------------------------
    
    print("\n" + "="*80)
    print("RANDOM FOREST RESULTS")
    print("="*80)
    
    print(f"\nDataset Information:")
    print(f"  Total samples: {len(df_processed)}")
    print(f"  Number of classes: {n_classes}")
    print(f"  Target column: '{target_column}'")
    print(f"  Number of features: {len(all_features)}")
    
    print(f"\nData Split:")
    print(f"  Training samples: {len(train_df)}")
    print(f"  Test samples: {len(test_df)}")
    
    print(f"\nTraining Set Performance:")
    print(f"  Accuracy: {train_results['accuracy']:.4f}")
    print(f"  F1 Weighted: {train_results['f1_weighted']:.4f}")
    print(f"  Recall Weighted: {train_results['recall_weighted']:.4f}")
    print(f"  Precision Weighted: {train_results['precision_weighted']:.4f}")
    
    print(f"\nTest Set Performance:")
    print(f"  Accuracy: {test_results['accuracy']:.4f}")
    print(f"  F1 Weighted: {test_results['f1_weighted']:.4f}")
    print(f"  Recall Weighted: {test_results['recall_weighted']:.4f}")
    print(f"  Precision Weighted: {test_results['precision_weighted']:.4f}")
    
    print(f"\nTop 10 Most Important Features:")
    top_features = feature_importances.head(10)
    for idx, row in top_features.iterrows():
        print(f"  {row['feature']}: {row['importance']:.4f}")
    
    print(f"\nConfusion Matrix (Test Set):")
    print(test_results['confusion_matrix'])
    
    print(f"\nClassification Report (Test Set):")
    print(test_results['classification_report'])
    
    print("\n" + "="*80)
    print("RANDOM FOREST PIPELINE COMPLETED")
    print("="*80)
    
    # Return all results
    return {
        'train_df': train_df,
        'test_df': test_df,
        'classifier': rf_classifier,
        'scaler': scaler,
        'train_results': train_results,
        'test_results': test_results,
        'feature_importances': feature_importances,
        'all_features': all_features,
        'n_classes': n_classes,
        'target_column': target_column,
        'preserve_order': preserve_order
    }

# ------------------------------------------------------------
# 2. MODE FUNCTIONS (Similar to your existing interface)
# ------------------------------------------------------------

def run_rf_mode_1_single_file(file_path, preserve_order=False, **kwargs):
    """
    MODE 1: Read a single CSV file and split into train/test internally
    """
    print("="*80)
    print("MODE 1: SINGLE FILE TRAIN/TEST SPLIT")
    print("="*80)
    print(f"Note: Order preservation = {preserve_order}")
    if preserve_order:
        print("      Using CLASS-WISE sequential split")
    
    df = pd.read_csv(file_path)
    print(f"✓ Loaded data from: {file_path}")
    print(f"✓ Data shape: {df.shape}")
    print(f"✓ Columns: {df.columns.tolist()}")
    
    # Ensure preserve_order is passed to pipeline
    kwargs['preserve_order'] = preserve_order
    
    results = run_standard_random_forest(df=df, **kwargs)
    return results

def run_rf_mode_2_two_files(train_file_path, test_file_path, **kwargs):
    """
    MODE 2: Read two separate CSV files for train and test
    """
    print("="*80)
    print("MODE 2: SEPARATE TRAIN/TEST FILES")
    print("="*80)
    print("Note: Order is  not preserved from original files")
    
    # Load data
    train_df = pd.read_csv(train_file_path)
    test_df = pd.read_csv(test_file_path)
    
    print(f"✓ Train file: {train_file_path}, Shape: {train_df.shape}")
    print(f"✓ Test file: {test_file_path}, Shape: {test_df.shape}")
    
    # Ensure both have the same columns
    train_cols = set(train_df.columns)
    test_cols = set(test_df.columns)
    
    if train_cols != test_cols:
        print("⚠ Warning: Train and test files have different columns!")
        print(f"  Train columns: {train_df.columns.tolist()}")
        print(f"  Test columns: {test_df.columns.tolist()}")
        
        # Keep only common columns
        common_cols = list(train_cols.intersection(test_cols))
        print(f"✓ Using common columns: {common_cols}")
        
        if len(common_cols) < 2:
            raise ValueError("✗ Not enough common columns between train and test files!")
        
        train_df = train_df[common_cols]
        test_df = test_df[common_cols]
    
    # Run pipeline with external test set
    results = run_standard_random_forest(
        df=train_df,
        external_test_df=test_df,
        preserve_order=False,  # Always preserve order for external test
        **kwargs
    )
    return results

def run_rf_mode_3_dataframes(train_df, test_df=None, preserve_order=True, **kwargs):
    """
    MODE 3: Use existing DataFrames
    """
    if test_df is None:
        print("="*80)
        print("MODE 3: SINGLE DATAFRAME")
        print("="*80)
        print(f"Note: Order preservation = {preserve_order}")
        if preserve_order:
            print("      Using CLASS-WISE sequential split")
        
        kwargs['preserve_order'] = preserve_order
        results = run_standard_random_forest(df=train_df, **kwargs)
    else:
        print("="*80)
        print("MODE 3: TWO DATAFRAMES")
        print("="*80)
        print("Note: Order is preserved from input DataFrames")
        
        results = run_standard_random_forest(
            df=train_df,
            external_test_df=test_df,
            preserve_order=False,
            **kwargs
        )
    return results


## CDR-MLC Test 1: level_1 and level_3

In [3]:
results = run_rf_mode_2_two_files(
    train_file_path='DATASETS/CDR-MLC/scale_1/Long/CDR-MLC-Shuffle.csv',
    test_file_path='DATASETS/CDR-MLC/scale_1/Short/CDR-MLC-Shuffle.csv',
    target_column='label'
    )

MODE 2: SEPARATE TRAIN/TEST FILES
Note: Order is  not preserved from original files
✓ Train file: DATASETS/CDR-MLC/scale_1/Long/CDR-MLC-Shuffle.csv, Shape: (258579, 40)
✓ Test file: DATASETS/CDR-MLC/scale_1/Short/CDR-MLC-Shuffle.csv, Shape: (7136, 39)
⚠ Warning: Train and test files have different columns!
  Train columns: ['pRetran', 'sMeanPktSz', 'SrcRetra', 'PCRatio', 'SrcWin', 'SrcLoss', 'DstRate', 'SrcLoad', 'Load', 'DstLoad', 'TcpRtt', 'Sum', 'AckDat', 'dTtl', 'Min', 'Max', 'pLoss', 'DstLoss', 'Loss', 'StdDev', 'Rate', 'SrcRate', 'IdleTime', 'Dur', 'SrcPkts', 'SrcGap', 'DstBytes', 'DstGap', 'sTtl', 'DstWin', 'TotPkts', 'DstPkts', 'Mean', 'SrcBytes', 'TotBytes', 'dMeanPktSz', 'DstRetra', 'SynAck', 'label', 'index_in_label']
  Test columns: ['pRetran', 'SrcRetra', 'PCRatio', 'SrcWin', 'SrcLoss', 'DstRate', 'SrcLoad', 'Load', 'DstLoad', 'TcpRtt', 'Sum', 'AckDat', 'dTtl', 'Min', 'pLoss', 'DstLoss', 'Loss', 'StdDev', 'Rate', 'SrcRate', 'IdleTime', 'Dur', 'SrcPkts', 'SrcGap', 'DstBytes

In [4]:
results = run_rf_mode_2_two_files(
    train_file_path='DATASETS/CDR-MLC/scale_1/Short/CDR-MLC-Shuffle.csv',
    test_file_path='DATASETS/CDR-MLC/scale_1/Long/CDR-MLC-Shuffle.csv',
    target_column='label'
    )

MODE 2: SEPARATE TRAIN/TEST FILES
Note: Order is  not preserved from original files
✓ Train file: DATASETS/CDR-MLC/scale_1/Short/CDR-MLC-Shuffle.csv, Shape: (7136, 39)
✓ Test file: DATASETS/CDR-MLC/scale_1/Long/CDR-MLC-Shuffle.csv, Shape: (258579, 40)
⚠ Warning: Train and test files have different columns!
  Train columns: ['pRetran', 'SrcRetra', 'PCRatio', 'SrcWin', 'SrcLoss', 'DstRate', 'SrcLoad', 'Load', 'DstLoad', 'TcpRtt', 'Sum', 'AckDat', 'dTtl', 'Min', 'pLoss', 'DstLoss', 'Loss', 'StdDev', 'Rate', 'SrcRate', 'IdleTime', 'Dur', 'SrcPkts', 'SrcGap', 'DstBytes', 'DstGap', 'sTtl', 'DstWin', 'TotPkts', 'DstPkts', 'Mean', 'SrcBytes', 'TotBytes', 'dMeanPktSz', 'DstRetra', 'SynAck', 'label', 'level_label', 'index_in_label_under_level']
  Test columns: ['pRetran', 'sMeanPktSz', 'SrcRetra', 'PCRatio', 'SrcWin', 'SrcLoss', 'DstRate', 'SrcLoad', 'Load', 'DstLoad', 'TcpRtt', 'Sum', 'AckDat', 'dTtl', 'Min', 'Max', 'pLoss', 'DstLoss', 'Loss', 'StdDev', 'Rate', 'SrcRate', 'IdleTime', 'Dur', 'Sr

## CDR-MLC Test 1: level_2 and level_3

In [5]:
results = run_rf_mode_2_two_files(
    train_file_path='DATASETS/CDR-MLC/scale_1/Short/level_1.csv',
    test_file_path='DATASETS/CDR-MLC/scale_1/Short/level_3.csv',
    target_column='label'
    )

MODE 2: SEPARATE TRAIN/TEST FILES
Note: Order is  not preserved from original files
✓ Train file: DATASETS/CDR-MLC/scale_1/Short/level_1.csv, Shape: (1930, 37)
✓ Test file: DATASETS/CDR-MLC/scale_1/Short/level_3.csv, Shape: (2263, 37)
STANDARD RANDOM FOREST PIPELINE

[1] DATA PREPARATION
    Target column: label
    Number of features: 34
    Total samples: 1930
    Number of classes: 5
    Class distribution:
      Class HTTP: 1188 samples (61.6%)
      Class SFTP: 112 samples (5.8%)
      Class SMTP: 193 samples (10.0%)
      Class SSH: 130 samples (6.7%)
      Class VIDEO: 307 samples (15.9%)

[2] DATA SPLITTING
    Using external test set provided
    Training samples: 1930
    Test samples: 2263

[3] DATA SCALING
    Features scaled using StandardScaler

[4] TRAINING RANDOM FOREST MODEL
    Model trained with 100 trees

[5] MODEL EVALUATION

RANDOM FOREST RESULTS

Dataset Information:
  Total samples: 1930
  Number of classes: 5
  Target column: 'label'
  Number of features: 34

D

## CDR-MLC Test 3: level_1 and level_2

In [8]:
results = run_rf_mode_2_two_files(
    train_file_path='DATASETS/CDR-MLC/scale_1/Short/level_1.csv',
    test_file_path='DATASETS/CDR-MLC/scale_1/Short/level_2.csv',
    target_column='label'
)

MODE 2: SEPARATE TRAIN/TEST FILES
Note: Order is  not preserved from original files
✓ Train file: DATASETS/CDR-MLC/scale_1/Short/level_1.csv, Shape: (1930, 37)
✓ Test file: DATASETS/CDR-MLC/scale_1/Short/level_2.csv, Shape: (2943, 37)
STANDARD RANDOM FOREST PIPELINE

[1] DATA PREPARATION
    Target column: label
    Number of features: 34
    Total samples: 1930
    Number of classes: 5
    Class distribution:
      Class HTTP: 1188 samples (61.6%)
      Class SFTP: 112 samples (5.8%)
      Class SMTP: 193 samples (10.0%)
      Class SSH: 130 samples (6.7%)
      Class VIDEO: 307 samples (15.9%)

[2] DATA SPLITTING
    Using external test set provided
    Training samples: 1930
    Test samples: 2943

[3] DATA SCALING
    Features scaled using StandardScaler

[4] TRAINING RANDOM FOREST MODEL
    Model trained with 100 trees

[5] MODEL EVALUATION

RANDOM FOREST RESULTS

Dataset Information:
  Total samples: 1930
  Number of classes: 5
  Target column: 'label'
  Number of features: 34

D

## ISCX-VPN

In [9]:
# ------------------------------------------------------------
# MAIN EXECUTION WITH COMMAND LINE INTERFACE SUPPORT
# ------------------------------------------------------------

    
# Example usage with your data
results = run_rf_mode_1_single_file(
    file_path='DATASETS/ISCX-VPN/ISCX-VPN.csv',
    target_column='label',
    test_size=0.2,
    preserve_order=True
)

MODE 1: SINGLE FILE TRAIN/TEST SPLIT
Note: Order preservation = True
      Using CLASS-WISE sequential split
✓ Loaded data from: DATASETS/ISCX-VPN/ISCX-VPN.csv
✓ Data shape: (44200, 37)
✓ Columns: ['pRetran', 'SrcRetra', 'PCRatio', 'SrcWin', 'SrcLoss', 'DstRate', 'SrcLoad', 'Load', 'DstLoad', 'TcpRtt', 'Sum', 'AckDat', 'dTtl', 'Min', 'pLoss', 'DstLoss', 'Loss', 'StdDev', 'Rate', 'SrcRate', 'IdleTime', 'Dur', 'SrcPkts', 'SrcGap', 'DstBytes', 'DstGap', 'sTtl', 'DstWin', 'TotPkts', 'DstPkts', 'Mean', 'SrcBytes', 'TotBytes', 'dMeanPktSz', 'DstRetra', 'SynAck', 'label']
STANDARD RANDOM FOREST PIPELINE

[1] DATA PREPARATION
    Target column: label
    Number of features: 34
    Total samples: 44200
    Number of classes: 6
    Class distribution:
      Class chat: 7119 samples (16.1%)
      Class file: 6159 samples (13.9%)
      Class mail: 650 samples (1.5%)
      Class p2p: 920 samples (2.1%)
      Class stream: 25675 samples (58.1%)
      Class voip: 3677 samples (8.3%)

[2] DATA SPLITTI

## ISCX-TOR

In [10]:
# ------------------------------------------------------------
# MAIN EXECUTION WITH COMMAND LINE INTERFACE SUPPORT
# ------------------------------------------------------------

    
# Example usage with your data
results = run_rf_mode_1_single_file(
    file_path='DATASETS/ISCX-TOR/ISCX-TOR.csv',
    target_column='label',
    test_size=0.2,
    preserve_order=True
)

MODE 1: SINGLE FILE TRAIN/TEST SPLIT
Note: Order preservation = True
      Using CLASS-WISE sequential split
✓ Loaded data from: DATASETS/ISCX-TOR/ISCX-TOR.csv
✓ Data shape: (14414, 37)
✓ Columns: ['pRetran', 'SrcRetra', 'PCRatio', 'SrcWin', 'SrcLoss', 'DstRate', 'SrcLoad', 'Load', 'DstLoad', 'TcpRtt', 'Sum', 'AckDat', 'dTtl', 'Min', 'pLoss', 'DstLoss', 'Loss', 'StdDev', 'Rate', 'SrcRate', 'IdleTime', 'Dur', 'SrcPkts', 'SrcGap', 'DstBytes', 'DstGap', 'sTtl', 'DstWin', 'TotPkts', 'DstPkts', 'Mean', 'SrcBytes', 'TotBytes', 'dMeanPktSz', 'DstRetra', 'SynAck', 'label']
STANDARD RANDOM FOREST PIPELINE

[1] DATA PREPARATION
    Target column: label
    Number of features: 34
    Total samples: 14414
    Number of classes: 8
    Class distribution:
      Class audio: 1035 samples (7.2%)
      Class browsing: 1887 samples (13.1%)
      Class chat: 502 samples (3.5%)
      Class ftp: 1665 samples (11.6%)
      Class mail: 461 samples (3.2%)
      Class p2p: 2138 samples (14.8%)
      Class vide